# Audio Instruct Evaluation
Clones [derkmed/audio_instruct_evaluation](https://github.com/derkmed/audio_instruct_evaluation), installs dependencies, and runs batched evaluation against the Universal-Audio-Understanding dataset.

**Recommended runtime:** A100 GPU (Runtime → Change runtime type)

## 1. Setup

In [ ]:
import os, sys

REPO_DIR = "/content/audio_instruct_evaluation"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/derkmed/audio_instruct_evaluation {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!pip install -q -r {REPO_DIR}/requirements.txt

## 2. Parameters
Edit these values — they replace the CLI flags from `main.py`.

In [ ]:
# @title Evaluation parameters { display-mode: "form" }

# --- Model ---
MODEL_CHOICE = "GEMMA-4"  # @param ["GEMMA-4", "QWEN3-Omni"]
MODEL_PATH_OVERRIDE = ""  # @param {type:"string"} — leave blank to use the default HF id

# --- Dataset ---
DATASET_NAME = "AudioInstruct/Universal-Audio-Understanding"  # @param {type:"string"}
DATASET_SPLIT = "test"  # @param {type:"string"}
JSON_CONFIG_PATH = "/content/audio_instruct_evaluation/clotho_config.json"  # @param {type:"string"}

# --- Inference ---
BATCH_SIZE = 4  # @param {type:"integer"}
NUM_PREPROCESSING_WORKERS = 4  # @param {type:"integer"}
MAX_NEW_TOKENS = 256  # @param {type:"integer"}
MAX_SAMPLES = 0  # @param {type:"integer"} — 0 means full dataset

# --- Output ---
OUTPUT_DIR = "/content/eval_results"  # @param {type:"string"} — leave blank to skip saving

# --- Google Drive ---
SAVE_TO_GDRIVE = False  # @param {type:"boolean"}
GDRIVE_FOLDER = "MyDrive/audio_instruct_results"  # @param {type:"string"} — path inside your Drive

# --- Auth ---
# Prefer storing your token in Colab Secrets (key: HF_TOKEN) rather than pasting it here.
HF_TOKEN_PARAM = ""  # @param {type:"string"}

In [ ]:
# Resolve HF token: Colab Secrets > param > env var
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = HF_TOKEN_PARAM or os.environ.get("HF_TOKEN") or None

print("HF token:", "set" if HF_TOKEN else "not set (may be required for gated models)")

## 3. Build config & backend

In [ ]:
from config import EvalConfig, DEFAULT_MODEL_PATHS
from backends import GemmaBackend, QwenBackend

config = EvalConfig(
    model_choice=MODEL_CHOICE,
    model_path=MODEL_PATH_OVERRIDE or None,
    dataset_name=DATASET_NAME,
    dataset_split=DATASET_SPLIT,
    json_config_path=JSON_CONFIG_PATH,
    batch_size=BATCH_SIZE,
    num_preprocessing_workers=NUM_PREPROCESSING_WORKERS,
    max_new_tokens=MAX_NEW_TOKENS,
    max_samples=MAX_SAMPLES if MAX_SAMPLES > 0 else None,
    output_dir=OUTPUT_DIR or None,
    hf_token=HF_TOKEN,
)

print(f"Model:   {config.resolved_model_path}")
print(f"Dataset: {config.dataset_name}  split={config.dataset_split}")
print(f"Samples: {config.max_samples or 'all'}  batch={config.batch_size}")

In [ ]:
backend_cls = {"GEMMA-4": GemmaBackend, "QWEN3-Omni": QwenBackend}[MODEL_CHOICE]
backend = backend_cls(config)

## 4. Load dataset

In [ ]:
from datasets import load_dataset

print(f"Loading {config.dataset_name} (split={config.dataset_split}) ...")
dataset = load_dataset(
    config.dataset_name,
    split=config.dataset_split,
    json_config_path=config.json_config_path,
    trust_remote_code=True,
    token=HF_TOKEN,
)
print(f"Loaded {len(dataset)} samples")

## 5. Run evaluation

In [ ]:
from evaluator import Evaluator

evaluator = Evaluator(backend, config)
results = evaluator.evaluate(dataset)

## 6. Results

In [ ]:
print("=== Final Results ===")
print(f"  WER:     {results['wer']:.4f}")
print(f"  Samples: {results['num_samples']}")

if config.output_dir:
    print(f"  Results saved to: {config.output_dir}/")

In [ ]:
# Preview the first few predictions
PREVIEW_N = 5  # @param {type:"integer"}

preds = results["predictions"][:PREVIEW_N]
refs  = results["references"][:PREVIEW_N]

for i, (p, r) in enumerate(zip(preds, refs)):
    print(f"[{i}] GT:   {r}")
    print(f"    Pred: {p}")
    print()

## 7. Save to Google Drive
Set `SAVE_TO_GDRIVE = True` and `GDRIVE_FOLDER` in the parameters cell above, then run this cell to mount your Drive and copy `results.jsonl` and `summary.json`.

In [ ]:
if SAVE_TO_GDRIVE:
    if not config.output_dir:
        print("OUTPUT_DIR is not set — nothing to copy to Drive.")
    else:
        import shutil
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)

        dest = f"/content/drive/{GDRIVE_FOLDER}"
        os.makedirs(dest, exist_ok=True)

        for filename in ("results.jsonl", "summary.json"):
            src = os.path.join(config.output_dir, filename)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(dest, filename))
                print(f"Copied {filename} → {dest}/")
            else:
                print(f"Skipped {filename} (not found in {config.output_dir})")
else:
    print("SAVE_TO_GDRIVE is False — skipping Drive upload.")

## 7. Export Results to Google Sheets

This section will convert the `results.jsonl` file into a spreadsheet and save it to Google Sheets.

In [ ]:
# Install gspread and gspread-dataframe
!pip install -q gspread gspread-dataframe

In [ ]:
import pandas as pd
import gspread
from gspread_dataframe import set_with_dataframe
import os
from google.colab import auth
from google.auth import default

# Ensure Google Drive is mounted if SAVE_TO_GDRIVE is True
if SAVE_TO_GDRIVE:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        print("Mounting Google Drive...")
        drive.mount("/content/drive", force_remount=True)
    else:
        print("Google Drive already mounted.")

# Path to the JSONL file
jsonl_file_path = os.path.join(config.output_dir, "results.jsonl")

# Read the JSONL file into a pandas DataFrame
try:
    df_results = pd.read_json(jsonl_file_path, lines=True)
    print(f"Successfully loaded {len(df_results)} records from {jsonl_file_path}")
except FileNotFoundError:
    print(f"Error: {jsonl_file_path} not found. Please ensure the evaluation ran successfully.")
    df_results = pd.DataFrame() # Create empty DataFrame to avoid errors

if not df_results.empty:
    # Authenticate user with Google Colab
    # This will prompt you to authorize access to your Google account.
    auth.authenticate_user()

    # Get authenticated credentials and authorize gspread
    creds, _ = default()
    gc = gspread.authorize(creds)

    # Define the spreadsheet name and folder
    spreadsheet_name = "AudioInstruct_Evaluation_Results"
    
    # Create or open the spreadsheet
    try:
        sh = gc.open(spreadsheet_name)
        print(f"Spreadsheet '{spreadsheet_name}' already exists. Opening it.")
    except gspread.exceptions.SpreadsheetNotFound:
        sh = gc.create(spreadsheet_name)
        print(f"Created new spreadsheet: '{spreadsheet_name}'")

    # Get the first worksheet or create one if it doesn't exist
    worksheet = sh.get_worksheet(0) if sh.worksheets() else sh.add_worksheet(title="Evaluation Results", rows="1", cols="1")

    # Clear existing content and set the DataFrame
    worksheet.clear()
    set_with_dataframe(worksheet, df_results)

    # Share the spreadsheet if needed (optional)
    # sh.share(email_address, perm_type='user', role='writer')

    print(f"Successfully exported data to Google Sheet: {sh.url}")

    if SAVE_TO_GDRIVE and GDRIVE_FOLDER:
        # Moving files to specific Drive folders after creation via gspread API is not straightforward.
        # The spreadsheet is created in your root Drive folder by default. 
        # You can manually move it from there if needed.
        print(f"Please manually move the spreadsheet to your desired Drive folder if needed: {GDRIVE_FOLDER}")
        print(f"The spreadsheet '{spreadsheet_name}' is available at: {sh.url}")
else:
    print("No data to export to Google Sheets.")